# Capitolo 14 — Ricorsione (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_java.ipynb)

Questo notebook accompagna il capitolo 14 «Ricorsione» delle dispense (volume 4, *Analisi e ricorsione*) e della [pagina del sito](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/14-ricorsione/). Contiene il codice **Java** degli algoritmi con pseudocodice del capitolo: `Factorial`, `Fib`, `IterativeFib`, `RecursiveGCD`, `FastPower`, `Hanoi`. Per ogni algoritmo ricorsivo il codice conta le **chiamate** (i nodi dell'albero delle chiamate), la **profondità** della ricorsione (l'altezza massima dello stack) e i **passi** (esecuzioni delle righe, numerate come nelle dispense: la riga 1 è l'intestazione `function`), e stampa la **traccia** dell'esecuzione: le chiamate in ordine di visita in profondità, con il rientro che indica la posizione nello stack. Gli esempi sono quelli del capitolo.

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime ▸ Esegui tutto* (oppure cella per cella con Maiusc+Invio). Non serve installare nulla.

Ogni programma è una cella `%%writefile` che scrive il file sorgente `.java`, seguita da una cella che lo compila con `javac` e lo esegue con `java`. La prima cella verifica (o installa) il JDK.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono campi `static` della classe, e la traccia si attiva con `trace = true`.

## Preparazione: il JDK

Su Colab `java` c'è già, ma serve anche il compilatore `javac`: la cella lo verifica e, se manca, installa il JDK.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## 1. Fattoriale: Factorial

**Input:** un numero naturale $n$. **Output:** $n!$, con $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ passi; $n + 1$ chiamate, profondità $n + 1$ (l'albero è un cammino).

Pseudocodice: sezione 14.4 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/14-ricorsione/).

In [2]:
%%writefile Factorial.java
public class Factorial
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Factorial(n) valori corretti solo fino a n = 20 (in C++ e Java i 64 bit traboccano oltre)
    static long factorial(int n)
    {
        push(n); cnt[2]++;
        if (n == 0)
        {
            cnt[3]++;
            if (trace) event("Factorial(0): n = 0 => caso base");
            pop();
            if (trace) event("Factorial(0) restituisce 1");
            return 1;
        }
        if (trace) event(String.format("Factorial(%d): %d != 0 => deve calcolare %d * Factorial(%d) e attende", n, n, n, n - 1));
        cnt[5]++;
        long f = factorial(n - 1);
        long r = n * f;
        pop();
        if (trace) event(String.format("Factorial(%d) restituisce %d * %d = %d", n, n, f, r));
        return r;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: Factorial(4), con lo stack
        int n = 4;
        reset();
        trace = true;
        long r = factorial(n);
        trace = false;
        System.out.printf("Factorial(%d) = %d\n", n, r);
        System.out.printf("Chiamate = %d (nodi dell'albero), profondita' della ricorsione = %d\n", calls, maxDepth);
        printSteps(2, 3, 5);
        System.out.printf("S(n) = 2n + 2 = %d\n", 2 * n + 2);

        // Chiamate, profondita' e passi al variare di n
        System.out.printf("\n");
        System.out.printf("  n |                   n! | chiamate | profondita' | S(n) | 2n + 2\n");
        int[] values = {0, 1, 2, 3, 4, 5, 10, 15, 20};
        for (int v : values)
        {
            reset();
            long f = factorial(v);
            System.out.printf("%3d | %20d | %8d | %11d | %4d | %d\n", v, f, calls, maxDepth, total(), 2 * v + 2);
        }
    }
}

Writing Factorial.java


In [3]:
!javac Factorial.java && java Factorial

 1. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [4]
 2. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [4, 3]
 3. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => caso base   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) restituisce 1   stack: [4, 3, 2, 1]
 7. Factorial(1) restituisce 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) restituisce 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) restituisce 3 * 2 = 6   stack: [4]
10. Factorial(4) restituisce 4 * 6 = 24   stack: []
Factorial(4) = 24
Chiamate = 5 (nodi dell'albero), profondita' della ricorsione = 5
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
Passi totali S = 10
S(n) = 2n + 2 = 10

  n |                   n! | chiamate | profondita' | S(n) | 2n + 2
  0 |                    1 |        1 |           1 | 

**Prova tu.** Cambia $n$ e guarda lo stack crescere e poi svuotarsi.

In [4]:
%%writefile ProvaFactorial.java
public class ProvaFactorial
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- cambia n (al massimo 20)
        Factorial.reset();
        Factorial.trace = true;
        long r = Factorial.factorial(n);
        Factorial.trace = false;
        System.out.printf("Factorial(%d) = %d, chiamate = %d, profondita' = %d, S = %d\n", n, r, Factorial.calls, Factorial.maxDepth, Factorial.total());
    }
}

Writing ProvaFactorial.java


In [5]:
!javac ProvaFactorial.java && java ProvaFactorial

 1. Factorial(6): 6 != 0 => deve calcolare 6 * Factorial(5) e attende   stack: [6]
 2. Factorial(5): 5 != 0 => deve calcolare 5 * Factorial(4) e attende   stack: [6, 5]
 3. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => caso base   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) restituisce 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) restituisce 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) restituisce 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) restituisce 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) restituisce 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) restituisce 5 * 24 = 120   stack: [6]
14. Factorial(6) restituisce 6 * 120

## 2. Numeri di Fibonacci: Fib

**Input:** un numero naturale $n$. **Output:** $F(n)$, con $F(0) = 0$, $F(1) = 1$ e $F(n) = F(n-1) + F(n-2)$. Algoritmo ricorsivo naive: $C(n) = 2F(n+1) - 1$ chiamate e $S(n) = 4F(n+1) - 2 = 2^{\Theta(n)}$ passi.

Pseudocodice: sezione 14.5 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/14-ricorsione/).

In [6]:
%%writefile Fib.java
public class Fib
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // Fib(n)
    static long fib(int n)
    {
        push(n); cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            if (trace) say(String.format("Fib(%d)", n), String.format("caso base, restituisce %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("Fib(%d)", n), String.format("chiama Fib(%d)", n - 1));
        long x = fib(n - 1);                       // le due chiamate sono separate solo per stampare la traccia
        if (trace) say(String.format("Fib(%d)", n), String.format("chiama Fib(%d)", n - 2));
        long y = fib(n - 2);
        cnt[5]++;
        if (trace) say(String.format("Fib(%d)", n), String.format("restituisce %d + %d = %d", x, y, x + y));
        pop();
        return x + y;
    }

    static long fibValue(int k)                    // F(k) con un ciclo, per controllare
    {
        long a = 0, b = 1;
        for (int i = 0; i < k; i++)
        {
            long t = a + b;
            a = b;
            b = t;
        }
        return a;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: l'esecuzione di Fib(4), in ordine di visita in profondita'
        int n = 4;
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        long r = fib(n);
        trace = false;
        System.out.printf("F(%d) = %d, chiamate C(%d) = %d = 2F(%d) - 1, profondita' = %d\n", n, r, n, calls, n + 1, maxDepth);

        // Fib(5): gli stessi sottoproblemi sono risolti piu' volte
        System.out.printf("\n");
        n = 5;
        reset();
        r = fib(n);
        System.out.printf("Fib(%d) = %d: %d chiamate, profondita' %d, S = %d passi\n", n, r, calls, maxDepth, total());
        System.out.printf("  k | chiamate di Fib(k)\n");
        for (int k = n; k >= 0; k--) System.out.printf("%3d | %d\n", k, seen[k]);
        printSteps(2, 3, 5);

        // Numero di chiamate e di passi: C(n) = 2F(n+1) - 1 e S(n) = 4F(n+1) - 2
        System.out.printf("\n");
        System.out.printf("  n |   F(n) |     C(n) | 2F(n+1) - 1 |     S(n) | 4F(n+1) - 2 | profondita'\n");
        int[] values = {0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = fib(v);
            System.out.printf("%3d | %6d | %8d | %11d | %8d | %11d | %d\n", v, r, calls, 2 * fibValue(v + 1) - 1, total(), 4 * fibValue(v + 1) - 2, maxDepth);
        }
    }
}

Writing Fib.java


In [7]:
!javac Fib.java && java Fib

chiamata                    | azione
Fib(4)                      | chiama Fib(3)
  Fib(3)                    | chiama Fib(2)
    Fib(2)                  | chiama Fib(1)
      Fib(1)                | caso base, restituisce 1
    Fib(2)                  | chiama Fib(0)
      Fib(0)                | caso base, restituisce 0
    Fib(2)                  | restituisce 1 + 0 = 1
  Fib(3)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(3)                    | restituisce 1 + 1 = 2
Fib(4)                      | chiama Fib(2)
  Fib(2)                    | chiama Fib(1)
    Fib(1)                  | caso base, restituisce 1
  Fib(2)                    | chiama Fib(0)
    Fib(0)                  | caso base, restituisce 0
  Fib(2)                    | restituisce 1 + 0 = 1
Fib(4)                      | restituisce 2 + 1 = 3
F(4) = 3, chiamate C(4) = 9 = 2F(5) - 1, profondita' = 4

Fib(5) = 5: 15 chiamate, profondita' 5, S = 30 passi
  k | chiamate di

**Prova tu.** Cambia $n$ e guarda quanto crescono le chiamate.

In [8]:
%%writefile ProvaFib.java
public class ProvaFib
{
    public static void main(String[] args)
    {
        int n = 6;        // <-- cambia n (con n > 45 ci vuole un po')
        Fib.reset();
        long r = Fib.fib(n);
        System.out.printf("F(%d) = %d, chiamate = %d, S = %d\n", n, r, Fib.calls, Fib.total());
    }
}

Writing ProvaFib.java


In [9]:
!javac ProvaFib.java && java ProvaFib

F(6) = 8, chiamate = 25, S = 50


## 3. Fibonacci iterativo: IterativeFib

**Input:** un numero naturale $n$. **Output:** $F(n)$, calcolato da $F(0)$ e $F(1)$ in su tenendo solo gli ultimi due valori. $S(n) = 5n + 1 = \Theta(n)$ passi. Invariante: prima di ogni valutazione della condizione, $prev = F(i-2)$ e $curr = F(i-1)$.

Pseudocodice: sezione 14.5.2 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/14-ricorsione/).

In [10]:
%%writefile IterativeFib.java
public class IterativeFib
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // IterativeFib(n)
    static long iterativeFib(int n, boolean trace)
    {
        cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            return n;
        }
        long prev = 0; cnt[5]++;
        long curr = 1; cnt[6]++;
        int i = 2; cnt[7]++;
        if (trace)                                 // traccia: riga di inizializzazione
            System.out.println(String.format("%-10s | %-8s | %-18s | %4d | %4d", "inizio", "2", "-", prev, curr));
        cnt[8]++;
        while (i <= n)
        {
            long next = prev + curr; cnt[9]++;
            prev = curr; cnt[10]++;
            curr = next; cnt[11]++;
            i = i + 1; cnt[12]++;
            cnt[8]++;
            if (trace)                             // traccia: i - 2 e' il numero dell'iterazione
                System.out.println(String.format("%-10d | %-8s | %-18s | %4d | %4d", i - 2, String.format("%d -> %d", i - 1, i), String.format("%d + %d = %d", next - prev, prev, next), prev, curr));
        }
        cnt[14]++;
        return curr;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: IterativeFib(4)
        int n = 4;
        reset();
        System.out.printf("iterazione | i        | next = prev + curr | prev | curr\n");
        long r = iterativeFib(n, true);
        System.out.printf("Il ciclo termina con i = %d > %d: restituisce curr = %d\n", n + 1, n, r);
        printSteps(2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14);
        System.out.printf("S(n) = 5n + 1 = %d\n", 5 * n + 1);

        // Confronto con Fib: passi di Fib = 4F(n+1) - 2 (formula verificata nella sezione precedente)
        System.out.printf("\n");
        System.out.printf("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1\n");
        int[] values = {5, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = iterativeFib(v, false);
            long steps = total();
            System.out.printf("%3d | %6d | %16d | %12d | %d\n", v, r, 4 * iterativeFib(v + 1, false) - 2, steps, 5 * v + 1);
        }
    }
}

Writing IterativeFib.java


In [11]:
!javac IterativeFib.java && java IterativeFib

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
Il ciclo termina con i = 5 > 4: restituisce curr = 3
Riga | Esecuzioni
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Passi totali S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Prova tu.** Cambia $n$ e controlla che i passi siano $5n + 1$.

In [12]:
%%writefile ProvaIterativeFib.java
public class ProvaIterativeFib
{
    public static void main(String[] args)
    {
        int n = 10;        // <-- cambia n
        IterativeFib.reset();
        System.out.printf("iterazione | i        | next = prev + curr | prev | curr\n");
        long r = IterativeFib.iterativeFib(n, true);
        System.out.printf("F(%d) = %d, S = %d = 5n + 1 = %d\n", n, r, IterativeFib.total(), 5 * n + 1);
    }
}

Writing ProvaIterativeFib.java


In [13]:
!javac ProvaIterativeFib.java && java ProvaIterativeFib

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


## 4. Massimo comun divisore: RecursiveGCD

**Input:** due numeri naturali $n \ge 1$ e $m \ge 0$. **Output:** $\gcd(n, m)$. Ricorsione di coda: ogni chiamata ne fa al più una, l'albero è un cammino. Al più $2\lfloor\log_2 m\rfloor + 3$ chiamate, $O(\log_2 m)$ passi.

Pseudocodice: sezione 14.6 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/14-ricorsione/).

In [14]:
%%writefile RecursiveGCD.java
public class RecursiveGCD
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static int floorLog2(int m)                    // floor(log2 m) con un ciclo (solo per il limite sulle chiamate)
    {
        int k = 0;
        while (m > 1)
        {
            m = m / 2;
            k = k + 1;
        }
        return k;
    }

    // RecursiveGCD(n, m)
    static int recursiveGCD(int n, int m)
    {
        push(m); cnt[2]++;
        if (m == 0)
        {
            cnt[3]++;
            if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("caso base, restituisce %d", n));
            pop();
            return n;
        }
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("%d mod %d = %d: chiama RecursiveGCD(%d, %d)", n, m, n % m, m, n % m));
        cnt[5]++;
        int g = recursiveGCD(m, n % m);
        if (trace) say(String.format("RecursiveGCD(%d, %d)", n, m), String.format("restituisce %d", g));
        pop();
        return g;
    }

    public static void main(String[] args)
    {
        // Esempi delle dispense: RecursiveGCD(48, 18) e RecursiveGCD(252, 105), alberi delle chiamate (cammini)
        int[][] pairs = {{48, 18}, {252, 105}};
        for (int[] p : pairs)
        {
            reset();
            trace = true;
            System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
            int g = recursiveGCD(p[0], p[1]);
            trace = false;
            System.out.printf("gcd(%d, %d) = %d, chiamate = %d, S = %d\n", p[0], p[1], g, calls, total());
            System.out.printf("\n");
        }

        // Numero di chiamate (ogni chiamata esegue 2 righe, quindi S = 2 chiamate) e limite 2 floor(log2 m) + 3
        System.out.printf("      n |       m |  gcd | chiamate |   S | 2 floor(log2 m) + 3\n");
        int[][] more = {{48, 18}, {252, 105}, {89, 55}, {144, 89}, {1000, 7}, {1000000, 999999}};
        for (int[] p : more)
        {
            reset();
            int g = recursiveGCD(p[0], p[1]);
            System.out.printf("%7d | %7d | %4d | %8d | %3d | %d\n", p[0], p[1], g, calls, total(), 2 * floorLog2(p[1]) + 3);
        }
    }
}

Writing RecursiveGCD.java


In [15]:
!javac RecursiveGCD.java && java RecursiveGCD

chiamata                    | azione
RecursiveGCD(48, 18)        | 48 mod 18 = 12: chiama RecursiveGCD(18, 12)
  RecursiveGCD(18, 12)      | 18 mod 12 = 6: chiama RecursiveGCD(12, 6)
    RecursiveGCD(12, 6)     | 12 mod 6 = 0: chiama RecursiveGCD(6, 0)
      RecursiveGCD(6, 0)    | caso base, restituisce 6
    RecursiveGCD(12, 6)     | restituisce 6
  RecursiveGCD(18, 12)      | restituisce 6
RecursiveGCD(48, 18)        | restituisce 6
gcd(48, 18) = 6, chiamate = 4, S = 8

chiamata                    | azione
RecursiveGCD(252, 105)      | 252 mod 105 = 42: chiama RecursiveGCD(105, 42)
  RecursiveGCD(105, 42)     | 105 mod 42 = 21: chiama RecursiveGCD(42, 21)
    RecursiveGCD(42, 21)    | 42 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(42, 21)    | restituisce 21
  RecursiveGCD(105, 42)     | restituisce 21
RecursiveGCD(252, 105)      | restituisce 21
gcd(252, 105) = 21, chiamate = 4, S = 8

      n |       m |  gcd | ch

**Prova tu.** Cambia i due numeri; il caso peggiore sono due numeri di Fibonacci consecutivi.

In [16]:
%%writefile ProvaRecursiveGCD.java
public class ProvaRecursiveGCD
{
    public static void main(String[] args)
    {
        int n = 1071, m = 462;        // <-- cambia i due numeri
        RecursiveGCD.reset();
        RecursiveGCD.trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        int g = RecursiveGCD.recursiveGCD(n, m);
        RecursiveGCD.trace = false;
        System.out.printf("gcd(%d, %d) = %d, chiamate = %d\n", n, m, g, RecursiveGCD.calls);
    }
}

Writing ProvaRecursiveGCD.java


In [17]:
!javac ProvaRecursiveGCD.java && java ProvaRecursiveGCD

chiamata                    | azione
RecursiveGCD(1071, 462)     | 1071 mod 462 = 147: chiama RecursiveGCD(462, 147)
  RecursiveGCD(462, 147)    | 462 mod 147 = 21: chiama RecursiveGCD(147, 21)
    RecursiveGCD(147, 21)   | 147 mod 21 = 0: chiama RecursiveGCD(21, 0)
      RecursiveGCD(21, 0)   | caso base, restituisce 21
    RecursiveGCD(147, 21)   | restituisce 21
  RecursiveGCD(462, 147)    | restituisce 21
RecursiveGCD(1071, 462)     | restituisce 21
gcd(1071, 462) = 21, chiamate = 4


## 5. Elevamento a potenza veloce: FastPower

**Input:** un numero $x$ e un numero naturale $n$. **Output:** $x^n$, per quadrati ripetuti. $S(n) = 4\lfloor\log_2 n\rfloor + 6 = \Theta(\log_2 n)$ passi, $\lfloor\log_2 n\rfloor + 2$ chiamate, al più $2(\lfloor\log_2 n\rfloor + 1)$ moltiplicazioni.

Pseudocodice: sezione 14.7 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/14-ricorsione/).

In [18]:
%%writefile FastPower.java
public class FastPower
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static int floorLog2(int n)                    // floor(log2 n) con un ciclo (solo per la formula dei passi)
    {
        int k = 0;
        while (n > 1)
        {
            n = n / 2;
            k = k + 1;
        }
        return k;
    }

    // FastPower(x, n)   i valori troppo grandi traboccano in C++ e Java, non in Python
    static long fastPower(long x, int n)
    {
        push(n); cnt[2]++;
        if (n == 0)
        {
            cnt[3]++;
            if (trace) say(String.format("FastPower(%d, 0)", x), "caso base, restituisce 1");
            pop();
            return 1;
        }
        if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("%d != 0 => chiama FastPower(%d, %d)   (%d/2 = %d)", n, x, n / 2, n, n / 2));
        long y = fastPower(x, n / 2); cnt[5]++;    // una sola chiamata ricorsiva: il risultato e' in y
        cnt[6]++;
        long r;
        if (n % 2 == 0)
        {
            cnt[7]++;
            mults += 1;
            r = y * y;
            if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("y = %d, %d e' pari => restituisce %d * %d = %d", y, n, y, y, r));
        }
        else
        {
            cnt[9]++;
            mults += 2;
            r = x * y * y;
            if (trace) say(String.format("FastPower(%d, %d)", x, n), String.format("y = %d, %d e' dispari => restituisce %d * %d * %d = %d", y, n, x, y, y, r));
        }
        pop();
        return r;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: FastPower(3, 13)
        long x = 3;
        int n = 13;
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        long r = fastPower(x, n);
        trace = false;
        System.out.printf("%d^%d = %d, chiamate = %d, moltiplicazioni = %d\n", x, n, r, calls, mults);
        printSteps(2, 3, 5, 6, 7, 9);
        System.out.printf("S(n) = 4 floor(log2 n) + 6 = %d\n", 4 * floorLog2(n) + 6);

        // Quante moltiplicazioni: circa 2 log2 n invece di n (x = 1, per non traboccare)
        System.out.printf("\n");
        System.out.printf("      n | chiamate | moltiplicazioni | S(n) | 4 floor(log2 n) + 6\n");
        int[] values = {1, 2, 13, 100, 1000, 1000000};
        for (int v : values)
        {
            reset();
            fastPower(1, v);
            System.out.printf("%7d | %8d | %15d | %4d | %d\n", v, calls, mults, total(), 4 * floorLog2(v) + 6);
        }
    }
}

Writing FastPower.java


In [19]:
!javac FastPower.java && java FastPower

chiamata                    | azione
FastPower(3, 13)            | 13 != 0 => chiama FastPower(3, 6)   (13/2 = 6)
  FastPower(3, 6)           | 6 != 0 => chiama FastPower(3, 3)   (6/2 = 3)
    FastPower(3, 3)         | 3 != 0 => chiama FastPower(3, 1)   (3/2 = 1)
      FastPower(3, 1)       | 1 != 0 => chiama FastPower(3, 0)   (1/2 = 0)
        FastPower(3, 0)     | caso base, restituisce 1
      FastPower(3, 1)       | y = 1, 1 e' dispari => restituisce 3 * 1 * 1 = 3
    FastPower(3, 3)         | y = 3, 3 e' dispari => restituisce 3 * 3 * 3 = 27
  FastPower(3, 6)           | y = 27, 6 e' pari => restituisce 27 * 27 = 729
FastPower(3, 13)            | y = 729, 13 e' dispari => restituisce 3 * 729 * 729 = 1594323
3^13 = 1594323, chiamate = 5, moltiplicazioni = 7
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
   6 |          4
   7 |          1
   9 |          3
Passi totali S = 18
S(n) = 4 floor(log2 n) + 6 = 18

      n | chiamate | moltiplicazioni | S(n) | 4 f

**Prova tu.** Cambia $x$ e $n$ e conta le moltiplicazioni.

In [20]:
%%writefile ProvaFastPower.java
public class ProvaFastPower
{
    public static void main(String[] args)
    {
        long x = 2;
        int n = 10;        // <-- cambia x e n (con x e n grandi C++ e Java traboccano)
        FastPower.reset();
        FastPower.trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        long r = FastPower.fastPower(x, n);
        FastPower.trace = false;
        System.out.printf("%d^%d = %d, moltiplicazioni = %d\n", x, n, r, FastPower.mults);
    }
}

Writing ProvaFastPower.java


In [21]:
!javac ProvaFastPower.java && java ProvaFastPower

chiamata                    | azione
FastPower(2, 10)            | 10 != 0 => chiama FastPower(2, 5)   (10/2 = 5)
  FastPower(2, 5)           | 5 != 0 => chiama FastPower(2, 2)   (5/2 = 2)
    FastPower(2, 2)         | 2 != 0 => chiama FastPower(2, 1)   (2/2 = 1)
      FastPower(2, 1)       | 1 != 0 => chiama FastPower(2, 0)   (1/2 = 0)
        FastPower(2, 0)     | caso base, restituisce 1
      FastPower(2, 1)       | y = 1, 1 e' dispari => restituisce 2 * 1 * 1 = 2
    FastPower(2, 2)         | y = 2, 2 e' pari => restituisce 2 * 2 = 4
  FastPower(2, 5)           | y = 4, 5 e' dispari => restituisce 2 * 4 * 4 = 32
FastPower(2, 10)            | y = 32, 10 e' pari => restituisce 32 * 32 = 1024
2^10 = 1024, moltiplicazioni = 6


## 6. Torri di Hanoi: Hanoi

**Input:** il numero di dischi $n \ge 1$, il piolo di partenza $from$, quello di arrivo $to$ e quello ausiliario $aux$. **Output:** la sequenza delle mosse che porta i dischi $1, \dots, n$ da $from$ a $to$. $M(n) = 2^n - 1$ mosse (ottime), $S(n) = 3 \cdot 2^n - 4 = \Theta(2^n)$ passi, profondità $n$.

Pseudocodice: sezione 14.8 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi-ricorsione/14-ricorsione/).

In [22]:
%%writefile Hanoi.java
public class Hanoi
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    static boolean showMoves = false;              // true: stampa le mosse

    static void move(int disk, String from, String to)   // "move disk d from from to to": conta la mossa e, se showMoves, la stampa
    {
        moves++;
        if (showMoves) System.out.println(String.format("%d. sposta il disco %d da %s a %s", moves, disk, from, to));
    }

    // Hanoi(n, from, to, aux)
    static void hanoi(int n, String from, String to, String aux)
    {
        push(n); cnt[2]++;
        String call = trace ? String.format("Hanoi(%d,%s,%s,%s)", n, from, to, aux) : "";
        if (n == 1)
        {
            cnt[3]++;
            move(1, from, to);
            if (trace) say(call, String.format("stampa: sposta il disco 1 da %s a %s, ritorna", from, to));
        }
        else
        {
            if (trace) say(call, String.format("chiama Hanoi(%d,%s,%s,%s)", n - 1, from, aux, to));
            cnt[5]++;
            hanoi(n - 1, from, aux, to);
            cnt[6]++;
            move(n, from, to);
            if (trace) say(call, String.format("stampa: sposta il disco %d da %s a %s, chiama Hanoi(%d,%s,%s,%s)", n, from, to, n - 1, aux, to, from));
            cnt[7]++;
            hanoi(n - 1, aux, to, from);
            if (trace) say(call, "ritorna");
        }
        pop();
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: Hanoi(3, A, C, B), le mosse
        reset();
        showMoves = true;
        hanoi(3, "A", "C", "B");
        showMoves = false;
        System.out.printf("%d mosse, %d chiamate, profondita' della ricorsione = %d\n", moves, calls, maxDepth);

        // La stessa esecuzione, chiamata per chiamata (rientro = profondita' nello stack)
        System.out.printf("\n");
        reset();
        trace = true;
        System.out.println(String.format("%-28s| %s", "chiamata", "azione"));
        hanoi(3, "A", "C", "B");
        trace = false;
        printSteps(2, 3, 5, 6, 7);
        System.out.printf("S(n) = 3 * 2^n - 4 = %d\n", 3 * (1 << 3) - 4);

        // Mosse e passi: M(n) = 2^n - 1 e S(n) = 3 * 2^n - 4
        System.out.printf("\n");
        System.out.printf("  n |  mosse | 2^n - 1 | chiamate |   S(n) | 3 * 2^n - 4 | profondita'\n");
        for (int k = 1; k <= 10; k++)
        {
            reset();
            hanoi(k, "A", "C", "B");
            System.out.printf("%3d | %6d | %7d | %8d | %6d | %11d | %d\n", k, moves, (1 << k) - 1, calls, total(), 3 * (1 << k) - 4, maxDepth);
        }

        // La leggenda: 64 dischi, una mossa al secondo
        System.out.printf("\n");
        System.out.printf("n = 64: 2^64 - 1 secondi sono piu' di %.1e anni\n", (Math.pow(2.0, 64) - 1) / (365.25 * 24 * 3600));
    }
}

Writing Hanoi.java


In [23]:
!javac Hanoi.java && java Hanoi

1. sposta il disco 1 da A a C
2. sposta il disco 2 da A a B
3. sposta il disco 1 da C a B
4. sposta il disco 3 da A a C
5. sposta il disco 1 da B a A
6. sposta il disco 2 da B a C
7. sposta il disco 1 da A a C
7 mosse, 7 chiamate, profondita' della ricorsione = 3

chiamata                    | azione
Hanoi(3,A,C,B)              | chiama Hanoi(2,A,B,C)
  Hanoi(2,A,B,C)            | chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)          | stampa: sposta il disco 1 da A a C, ritorna
  Hanoi(2,A,B,C)            | stampa: sposta il disco 2 da A a B, chiama Hanoi(1,C,B,A)
    Hanoi(1,C,B,A)          | stampa: sposta il disco 1 da C a B, ritorna
  Hanoi(2,A,B,C)            | ritorna
Hanoi(3,A,C,B)              | stampa: sposta il disco 3 da A a C, chiama Hanoi(2,B,C,A)
  Hanoi(2,B,C,A)            | chiama Hanoi(1,B,A,C)
    Hanoi(1,B,A,C)          | stampa: sposta il disco 1 da B a A, ritorna
  Hanoi(2,B,C,A)            | stampa: sposta il disco 2 da B a C, chiama Hanoi(1,A,C,B)
    Hanoi(1,A,C,B)

**Prova tu.** Cambia il numero di dischi e conta le mosse.

In [24]:
%%writefile ProvaHanoi.java
public class ProvaHanoi
{
    public static void main(String[] args)
    {
        int n = 4;        // <-- cambia il numero di dischi
        Hanoi.reset();
        Hanoi.showMoves = true;
        Hanoi.hanoi(n, "A", "C", "B");
        Hanoi.showMoves = false;
        System.out.printf("%d mosse = 2^%d - 1 = %d\n", Hanoi.moves, n, (1 << n) - 1);
    }
}

Writing ProvaHanoi.java


In [25]:
!javac ProvaHanoi.java && java ProvaHanoi

1. sposta il disco 1 da A a B
2. sposta il disco 2 da A a C
3. sposta il disco 1 da B a C
4. sposta il disco 3 da A a B
5. sposta il disco 1 da C a A
6. sposta il disco 2 da C a B
7. sposta il disco 1 da A a B
8. sposta il disco 4 da A a C
9. sposta il disco 1 da B a C
10. sposta il disco 2 da B a A
11. sposta il disco 1 da C a A
12. sposta il disco 3 da B a C
13. sposta il disco 1 da A a B
14. sposta il disco 2 da A a C
15. sposta il disco 1 da B a C
15 mosse = 2^4 - 1 = 15
